# Лабораторная работа: Positional Embeddings руками

Цель: разобраться, зачем трансформеру нужны positional embeddings, и реализовать ключевые части самостоятельно: индексы позиций, sinusoidal frequencies, матрицу positional encoding, добавление к batch, метрики похожести, относительный сдвиг и toy RoPE.

Важное правило этой лабораторной: мы **не импортируем готовые функции** из `transformer_lab`. Все вычисления ниже делаются руками на PyTorch.

## Learning outcomes

После выполнения лабораторной студент должен уметь:

- объяснить, почему self-attention без позиционного сигнала permutation equivariant;
- вывести sinusoidal positional encoding из частот `omega_i`;
- реализовать PE без готовых библиотечных функций;
- проверять корректность реализации через shape invariants, numerical error и qualitative markers;
- объяснить свойство относительного сдвига через формулы `sin(a+b)` и `cos(a+b)`;
- отличать additive sinusoidal PE от RoPE на уровне того, **куда** вводится позиционная информация;
- провести маленький исследовательский эксперимент и защитить вывод метриками.

**Deliverables:** заполненные TODO-ячейки, пройденные проверки, краткий отчет по вопросам после блоков, один самостоятельный эксперимент.

## Prerequisites

Перед началом полезно уверенно понимать:

- формы тензоров в PyTorch;
- broadcasting;
- `unsqueeze`, slicing, `repeat` или broadcasting-эквиваленты;
- dot product и cosine similarity;
- базовые свойства `sin`, `cos` и формулы сложения углов.

Если что-то из этого проседает, это нормально: в лабораторной проверки специально построены так, чтобы ошибка формы или формулы проявлялась быстро.

## Как работать с этой лабораторной

Каждый блок устроен одинаково:

1. **Разбор идеи**: короткая теория и минимальный пример.
2. **Прогноз**: перед кодом попробуйте словами предсказать форму тензора или поведение метрики.
3. **Мини-лаба**: ячейка с `# здесь ваш код`.
4. **Проверка**: отдельная ячейка с маркерами корректности.
5. **Вывод**: одна-две фразы в отчете, что именно стало понятно.

Лучше не запускать весь notebook целиком до заполнения TODO-ячеек: проверки специально падают, если код еще не написан.

## Как устроены проверки

Проверочные ячейки не используют готовую реализацию из проекта. Они проверяют независимые свойства:

- **Shape invariants:** правильные размерности тензоров.
- **Boundary cases:** позиция `0`, нечетный `d_model`, выход за `max_len`.
- **Formula checks:** отдельная координата сравнивается с ручным `math.sin` / `math.cos`.
- **Metric checks:** cosine similarity, max absolute error, norm preservation.
- **Qualitative markers:** ненулевые позиции после RoPE должны измениться, а позиция `0` нет.

Такой стиль проверки похож на mini-autograder: он не диктует конкретный код, но ловит неправильную математику.

## Что берем из лекции

В статье Suvash Sedhain материал раскрывается через несколько удачных учебных приемов:

- сначала показывается **конкретная проблема порядка** на двух предложениях с одинаковыми токенами;
- формула сразу связывается с **визуальной картинкой волн**: быстрые и медленные частоты;
- относительная позиция объясняется как **поворот на циферблате**;
- RoPE сначала разбирается в **2D**, а затем переносится на пары измерений;
- long-range behavior показывается как **кривая decay**, а не только как текстовое утверждение.

В этой лабораторной эти приемы превращены в интерактивные блоки: маленькие визуализации, прогнозы перед кодом и проверки после кода.

## Notation

| Symbol | Meaning |
|---|---|
| `B` | batch size |
| `T` / `seq_len` | sequence length |
| `D` / `d_model` | embedding width |
| `pos` | absolute token position |
| `i` | sine/cosine pair index |
| `omega_i` | frequency for pair `i` |
| `PE` | positional encoding matrix with shape `(max_len, d_model)` |

В этой лабораторной мы постоянно проверяем три вещи: **shape**, **численную точность**, **смысловой маркер**. Это ближе к реальной исследовательской работе, чем просто “код запустился”.

## 0. Подготовка

В ноутбуке нужен только PyTorch. Все функции positional embeddings вы будете писать сами.

In [ ]:
from __future__ import annotations

import math

import torch
from torch import Tensor
from torch.nn import functional as F


# Фиксируем seed, чтобы случайные тензоры и примеры были воспроизводимыми.
torch.manual_seed(7)

# Настраиваем печать тензоров: 4 знака после запятой, без scientific notation.
torch.set_printoptions(precision=4, sci_mode=False)

print("PyTorch:", torch.__version__)


def report_check(name: str, value: float, threshold: float | None = None) -> None:
    """Печатает результат проверки в компактном формате.

    Используется в проверочных ячейках, чтобы студент сразу видел:
    - название метрики;
    - полученное значение;
    - порог, если он есть;
    - статус OK/FAIL.
    """

    # Если порога нет, это просто диагностическая метрика.
    if threshold is None:
        print(f"{name}: {value:.4f}")
    else:
        # Если порог есть, сравниваем значение с ним и печатаем статус.
        status = "OK" if value <= threshold else "FAIL"
        print(f"{name}: {value:.2e} <= {threshold:.2e} [{status}]")


def preview_tensor(name: str, tensor: Tensor, rows: int = 4, cols: int = 8) -> None:
    """Показывает форму тензора и маленький верхний левый фрагмент.

    Это нужно, чтобы быстро проверить не весь большой тензор, а только
    его размерность и первые значения.
    """

    # Сначала печатаем shape: в задачах с embeddings ошибки формы встречаются часто.
    print(f"{name} shape: {tuple(tensor.shape)}")

    # Затем печатаем первые rows строк и cols столбцов как быстрый preview.
    print(tensor[:rows, :cols])


def tiny_heatmap(tensor: Tensor, rows: int = 12, cols: int = 16) -> None:
    """Рисует маленькую ASCII heatmap для численного тензора.

    `matplotlib` здесь не нужен: значения переводятся в символы от темных
    к ярким. Это помогает увидеть полосы и частоты в positional encoding.
    """

    # Набор символов от "маленького значения" к "большому значению".
    chars = " .:-=+*#%@"

    # Берем только небольшой фрагмент, чтобы вывод оставался читаемым.
    # detach() убирает связь с графом autograd, cpu() гарантирует печать на CPU.
    view = tensor[:rows, :cols].detach().cpu()

    # Находим диапазон значений в выбранном фрагменте.
    lo = view.min().item()
    hi = view.max().item()

    # Нормируем значения в диапазон [0, 1]. Маленькая 1e-12 защищает от деления на 0.
    scaled = (view - lo) / (hi - lo + 1e-12)

    # Каждое число заменяем символом с соответствующей "яркостью".
    for row in scaled:
        line = "".join(
            chars[min(int(v.item() * (len(chars) - 1)), len(chars) - 1)]
            for v in row
        )
        print(line)


def token_strip(tokens: list[str], highlight: dict[int, str] | None = None) -> str:
    """Возвращает строку с токенами и их позициями.

    Пример: `[0:the] [1:cat*] ...`. Это визуальный маркер того,
    что один и тот же токен может стоять на разных позициях.
    """

    # Если highlight не передали, используем пустой словарь.
    # В highlight можно передать, например, {1: "*"}, чтобы отметить позицию 1.
    highlight = highlight or {}

    boxes = []
    for i, token in enumerate(tokens):
        # Для каждой позиции достаем маркер, если он есть.
        marker = highlight.get(i, "")

        # Собираем компактный "бокс": индекс позиции + текст токена + маркер.
        boxes.append(f"[{i}:{token}{marker}]")

    # Склеиваем все боксы в одну строку для печати.
    return " ".join(boxes)


def dot_attention_scores(x: Tensor) -> Tensor:
    """Считает toy dot-product attention scores без softmax.

    Если `x` имеет форму `(seq_len, d_model)`, результат имеет форму
    `(seq_len, seq_len)`: каждая ячейка показывает dot product пары токенов.
    """

    # x @ x.T сравнивает каждый токен с каждым токеном через скалярное произведение.
    return x @ x.T


def print_square_matrix(name: str, matrix: Tensor, labels: list[str]) -> None:
    """Красиво печатает квадратную матрицу с подписями строк и столбцов.

    Используется для toy attention scores, чтобы было видно, какие токены
    имеют высокий dot product друг с другом.
    """

    print(name)

    # Печатаем заголовок: первые 4 символа каждого label, выровненные по ширине.
    header = "      " + " ".join(f"{label[:4]:>7}" for label in labels)
    print(header)

    # Печатаем строки матрицы: label строки + значения с двумя знаками после запятой.
    for label, row in zip(labels, matrix):
        values = " ".join(f"{value.item():7.2f}" for value in row)
        print(f"{label[:4]:>4}  {values}")


def ascii_curve(values: dict[int, float], width: int = 32) -> None:
    """Печатает простой ASCII-график для словаря численных значений.

    Ключ обычно означает distance или offset, а значение - метрику.
    Длина полоски пропорциональна значению метрики.
    """

    # Находим минимум и максимум, чтобы масштабировать все значения в один диапазон.
    lo = min(values.values())
    hi = max(values.values())

    for key, value in values.items():
        # Переводим значение в длину полоски от 0 до width.
        n = int((value - lo) / (hi - lo + 1e-12) * width)

        # Печатаем ключ, полоску и само число, чтобы график был и визуальным, и точным.
        print(f"{key:>4}: " + "#" * n + f" {value:.4f}")


## 1. Зачем вообще нужна позиция

Self-attention без позиционных признаков не знает порядок токенов. Если у модели есть только token embeddings, то перестановка токенов меняет порядок строк в тензоре, но не сообщает модели, какая строка была первой, второй или последней.

Самая простая идея: создать для каждой позиции отдельный вектор той же ширины, что и token embedding, и прибавить его:

```text
x_with_pos = token_embedding + position_vector
```

Так форма входа остается прежней: `(batch, seq_len, d_model)`.

> **Ключевой факт: permutation equivariance.**
>
> Если к входу self-attention не добавить позиционный сигнал, то перестановка токенов приводит к такой же перестановке выходов. Модель может сравнивать content токенов, но не имеет отдельной координаты “где токен стоял”.
>
> Поэтому positional encoding не просто косметическая добавка, а способ нарушить симметрию перестановок.

In [ ]:
batch_size = 2
seq_len = 6
d_model = 8

token_embeddings = torch.randn(batch_size, seq_len, d_model)
print("token embeddings shape:", tuple(token_embeddings.shape))
print("first token vector:", token_embeddings[0, 0])


### Визуальный блок: порядок токенов

Как в статье, начнем с двух фраз с одинаковым набором токенов. Без позиционного сигнала модель получает embeddings токенов, но не получает отдельного признака “это позиция 1”, “это позиция 5”.

Ниже видно: набор токенов одинаковый, но смысл отличается. Это и есть мотивация добавить позицию.

In [ ]:
sentence_a = ["the", "cat", "sat", "on", "the", "mat"]
sentence_b = ["the", "mat", "sat", "on", "the", "cat"]

print("A:", token_strip(sentence_a, {1: "*", 5: "*"}))
print("B:", token_strip(sentence_b, {1: "*", 5: "*"}))
print("same multiset of tokens:", sorted(sentence_a) == sorted(sentence_b))
print("same sequence:", sentence_a == sentence_b)


### Игрушечный attention-маркер без позиции

Сделаем детерминированные embeddings для слов и посчитаем dot-product scores. Без positional encoding строка с `cat` имеет тот же embedding независимо от того, стоит `cat` на позиции 1 или 5.

Это не полноценный transformer, а маленький маркер проблемы: content есть, coordinate нет.

In [ ]:
vocab = {"the": 0, "cat": 1, "sat": 2, "on": 3, "mat": 4}
word_vectors = torch.eye(len(vocab))

def embed_sentence(tokens: list[str]) -> Tensor:
    ids = torch.tensor([vocab[token] for token in tokens])
    return word_vectors[ids]

emb_a = embed_sentence(sentence_a)
emb_b = embed_sentence(sentence_b)

scores_a = dot_attention_scores(emb_a)
scores_b = dot_attention_scores(emb_b)

print_square_matrix("Scores A", scores_a, sentence_a)
print()
print_square_matrix("Scores B", scores_b, sentence_b)
print()
print("cat vector at pos 1 in A equals cat vector at pos 5 in B:", torch.allclose(emb_a[1], emb_b[5]))


### Мини-лабораторная 1: простые позиционные векторы

Сделайте простейшие positional vectors руками: создайте `position_ids = [0, 1, ..., seq_len - 1]`, превратите их в тензор формы `(seq_len, 1)`, затем повторите по последней размерности до `(seq_len, d_model)`.

После этого прибавьте positional vectors к `token_embeddings`. Это еще не sinusoidal PE, а разминка про формы и broadcasting.

In [ ]:
# Мини-лабораторная 1
# Задача: создать simple_position_vectors и simple_position_aware.

simple_position_vectors = None
simple_position_aware = None

# здесь ваш код


<details>
<summary>Подсказка</summary>

`torch.arange(seq_len)` дает позиции. После `unsqueeze(1)` получится колонка `(seq_len, 1)`. Повторить ее до `d_model` можно через `.repeat(1, d_model)` или broadcasting.
</details>

**После проверки:** запишите, почему `simple_position_vectors.unsqueeze(0)` можно прибавить ко всему batch.

In [ ]:
# Проверка мини-лабораторной 1
assert simple_position_vectors is not None, "Создайте simple_position_vectors"
assert simple_position_aware is not None, "Создайте simple_position_aware"
assert tuple(simple_position_vectors.shape) == (seq_len, d_model)
assert tuple(simple_position_aware.shape) == tuple(token_embeddings.shape)

expected_first_position = torch.zeros(d_model)
expected_last_position = torch.full((d_model,), seq_len - 1, dtype=simple_position_vectors.dtype)
assert torch.allclose(simple_position_vectors[0], expected_first_position)
assert torch.allclose(simple_position_vectors[-1], expected_last_position)

max_error = (simple_position_aware - (token_embeddings + simple_position_vectors.unsqueeze(0))).abs().max().item()
print(f"broadcast addition max error: {max_error:.2e}")
assert max_error < 1e-7


## 2. Sinusoidal positional encoding: частоты

В sinusoidal PE каждая пара измерений `(2i, 2i + 1)` использует свою частоту:

```text
omega_i = 1 / base^(2i / d_model)
```

Обычно `base = 10000`. Быстрые частоты меняются заметно уже на соседних позициях, медленные частоты кодируют более длинные масштабы.

> **Design choice: geometric frequency scale.**
>
> Частоты выбираются геометрически, а не линейно: `1, 1/base^(2/D), 1/base^(4/D), ...`.
> Это дает одновременно быстрые координаты для локальных различий и медленные координаты для длинного контекста.
>
> Вопрос исследовательского уровня: что меняется, если заменить геометрическую сетку частот на линейную?

### Мини-лабораторная 2: посчитайте frequency grid

Напишите функцию `make_frequencies`. Она должна вернуть тензор частот длины `d_model / 2`: по одной частоте на каждую sine/cosine пару.

Требования:

- `d_model` должен быть положительным;
- `d_model` должен быть четным;
- формула должна соответствовать `1 / base^(2i / d_model)`.

In [ ]:
# Мини-лабораторная 2
# Задача: реализовать частоты для sinusoidal positional encoding.

def make_frequencies(d_model: int, base: float = 10_000.0) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


<details>
<summary>Подсказка</summary>

Нужны индексы `0, 2, 4, ...`. Удобно взять `torch.arange(0, d_model, 2, dtype=torch.float32)`. Формула: `base ** (-indices / d_model)`.
</details>

**После проверки:** объясните, почему для `d_model=8` частоты равны примерно `[1, 0.1, 0.01, 0.001]`.

In [ ]:
# Проверка мини-лабораторной 2
freq = make_frequencies(8)
assert tuple(freq.shape) == (4,)
assert freq.dtype.is_floating_point

expected = torch.tensor([1.0, 0.1, 0.01, 0.001])
max_error = (freq - expected).abs().max().item()
print("frequencies:", freq)
report_check("frequency max error", max_error, 1e-6)
assert max_error < 1e-6

try:
    make_frequencies(7)
except ValueError:
    print("odd d_model check: ok")
else:
    raise AssertionError("Для нечетного d_model нужно выбросить ValueError")


### Визуальный блок: быстрые и медленные частоты

Статья использует образ “clock hands ticking at different speed”. Посчитаем период каждой пары измерений: чем выше период, тем медленнее меняется соответствующая пара sin/cos.

In [ ]:
freq = make_frequencies(16)
periods = 2 * math.pi / freq
for pair, (omega, period) in enumerate(zip(freq, periods)):
    bar_len = min(int(math.log10(period.item() + 1) * 8), 40)
    print(f"pair {pair:>2}: omega={omega.item():.6f}, period={period.item():>9.1f} " + "#" * bar_len)


## 3. Sinusoidal positional encoding: матрица PE

Теперь соберем полную матрицу:

```text
PE(pos, 2i)     = sin(pos * omega_i)
PE(pos, 2i + 1) = cos(pos * omega_i)
```

Эквивалентная запись из статьи:

```text
PE(pos, 2i)     = sin(pos / 10000^(2i / d_model))
PE(pos, 2i + 1) = cos(pos / 10000^(2i / d_model))
```

Форма матрицы: `(max_len, d_model)`.

> **Invariant: каждая sine/cosine пара лежит на единичной окружности.**
>
> Для каждой пары измерений:
>
> ```text
> sin^2(pos * omega_i) + cos^2(pos * omega_i) = 1
> ```
>
> Это важный sanity check: позиция меняет угол, но не длину каждой 2D-пары.

### Мини-лабораторная 3: реализуйте sinusoidal PE

Напишите `make_sinusoidal_pe`. Используйте вашу `make_frequencies`, создайте позиции, посчитайте углы и заполните четные/нечетные измерения.

In [ ]:
# Мини-лабораторная 3
# Задача: реализовать матрицу sinusoidal positional encoding.

def make_sinusoidal_pe(max_len: int, d_model: int, base: float = 10_000.0) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


<details>
<summary>Подсказка</summary>

Создайте `positions` формы `(max_len, 1)` и `frequencies` формы `(d_model / 2,)`. Их произведение даст `angles` формы `(max_len, d_model / 2)`. Потом заполните `pe[:, 0::2]` и `pe[:, 1::2]`.
</details>

**После проверки:** посмотрите на `PE(0)` и объясните, почему sin-координаты равны 0, а cos-координаты равны 1.

In [ ]:
# Проверка мини-лабораторной 3
pe = make_sinusoidal_pe(max_len=8, d_model=8)
assert tuple(pe.shape) == (8, 8)

print("PE(0):", pe[0])
assert torch.allclose(pe[0, 0::2], torch.zeros(4), atol=1e-7)
assert torch.allclose(pe[0, 1::2], torch.ones(4), atol=1e-7)

pos = 3
pair_index = 1
angle = pos / (10_000 ** (2 * pair_index / 8))
expected_sin = math.sin(angle)
expected_cos = math.cos(angle)

sin_error = abs(pe[pos, 2 * pair_index].item() - expected_sin)
cos_error = abs(pe[pos, 2 * pair_index + 1].item() - expected_cos)
report_check("manual coordinate sin error", sin_error, 1e-6)
report_check("manual coordinate cos error", cos_error, 1e-6)
assert sin_error < 1e-6
assert cos_error < 1e-6

# Дополнительный invariant: каждая пара (sin, cos) должна иметь норму 1.
pair_norms = pe[..., 0::2] ** 2 + pe[..., 1::2] ** 2
pair_norm_error = (pair_norms - 1.0).abs().max().item()
report_check("unit circle pair norm error", pair_norm_error, 1e-6)
assert pair_norm_error < 1e-6


### Быстрый визуальный маркер PE

Эта ячейка не требует `matplotlib`: она печатает маленькую ASCII-карту значений PE. Полосы и изменения яркости помогают увидеть, что разные измерения имеют разные частоты.

In [ ]:
preview_tensor("PE preview", pe, rows=6, cols=8)
print("\nTiny heatmap:")
tiny_heatmap(pe, rows=8, cols=8)


## 4. Добавление PE к batch

Token embeddings имеют форму `(batch, seq_len, d_model)`, а матрица PE имеет форму `(max_len, d_model)`. Для batch нужно выбрать первые `seq_len` строк PE и добавить batch dimension:

```text
PE[:seq_len].unsqueeze(0) -> (1, seq_len, d_model)
```

PyTorch сам применит broadcasting по batch dimension.

### Мини-лабораторная 4: напишите add_positional_encoding

Реализуйте функцию `add_positional_encoding(x, pe, start_pos=0)`. Она должна выбрать правильный срез `pe` и прибавить его к `x`.

`start_pos` пригодится для autoregressive decoding, когда текущий фрагмент начинается не с позиции `0`.

In [ ]:
# Мини-лабораторная 4
# Задача: добавить positional encoding к batch тензору.

def add_positional_encoding(x: Tensor, pe: Tensor, start_pos: int = 0) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


<details>
<summary>Подсказка</summary>

`seq_len = x.shape[1]`. Нужный срез: `pe[start_pos:start_pos + seq_len]`. Перед сложением добавьте batch dimension через `.unsqueeze(0)`.
</details>

**После проверки:** сформулируйте, зачем `start_pos` нужен при генерации текста по кускам.

In [ ]:
# Проверка мини-лабораторной 4
torch.manual_seed(17)
x = torch.randn(3, 5, 8)
pe = make_sinusoidal_pe(max_len=16, d_model=8)

out0 = add_positional_encoding(x, pe, start_pos=0)
out4 = add_positional_encoding(x, pe, start_pos=4)

assert tuple(out0.shape) == tuple(x.shape)
assert tuple(out4.shape) == tuple(x.shape)

error0 = (out0 - (x + pe[:5].unsqueeze(0))).abs().max().item()
error4 = (out4 - (x + pe[4:9].unsqueeze(0))).abs().max().item()
report_check("start_pos=0 max error", error0, 1e-7)
report_check("start_pos=4 max error", error4, 1e-7)
assert error0 < 1e-7
assert error4 < 1e-7

try:
    add_positional_encoding(x, pe, start_pos=20)
except ValueError:
    print("out of range check: ok")
else:
    raise AssertionError("Если срез выходит за max_len, нужно выбросить ValueError")


## 5. Метрика похожести позиций

Чтобы исследовать positional embeddings, можно сравнивать `PE(pos)` и `PE(pos + distance)` через cosine similarity. Если расстояние маленькое, векторы обычно более похожи. Для больших расстояний картина становится сложнее, потому что разные частоты имеют разные периоды.

### Мини-лабораторная 5: similarity curve

Реализуйте функцию `similarity_by_distance`, которая возвращает словарь `{distance: mean_cosine_similarity}` для списка расстояний.

In [ ]:
# Мини-лабораторная 5
# Задача: посчитать среднее cosine similarity между PE(pos) и PE(pos + distance).

def similarity_by_distance(pe: Tensor, distances: list[int]) -> dict[int, float]:
    # здесь ваш код
    raise NotImplementedError


<details>
<summary>Подсказка</summary>

Для distance `d`: сравните `pe[:-d]` и `pe[d:]`. `F.cosine_similarity(left, right, dim=-1)` вернет похожесть для каждой пары позиций. Потом возьмите `.mean().item()`.
</details>

**После проверки:** найдите расстояние, где похожесть перестает монотонно уменьшаться, и подумайте, почему частоты дают такую картину.

In [ ]:
# Проверка мини-лабораторной 5
pe = make_sinusoidal_pe(max_len=128, d_model=32)
distances = [1, 2, 4, 8, 16, 32]
curve = similarity_by_distance(pe, distances)

assert set(curve) == set(distances)
for distance, value in curve.items():
    print(f"distance={distance:>2}: mean cosine similarity={value:.4f}")
    assert isinstance(value, float)
    assert -1.0 <= value <= 1.0

assert curve[1] > curve[16], "Обычно соседние позиции должны быть похожее, чем позиции на расстоянии 16"
assert len(set(round(v, 4) for v in curve.values())) > 2, "Кривая не должна быть константой"

# Дополнительная независимая проверка одной точки кривой.
manual_similarity_d1 = F.cosine_similarity(pe[:-1], pe[1:], dim=-1).mean().item()
report_check("distance=1 independent recompute diff", abs(curve[1] - manual_similarity_d1), 1e-7)
assert abs(curve[1] - manual_similarity_d1) < 1e-7


### Визуальный блок: кривая похожести

Вместо таблицы чисел полезно увидеть форму зависимости. Это быстрый текстовый график: чем длиннее строка, тем выше cosine similarity.

In [ ]:
ascii_curve(curve)


## 6. Относительный сдвиг как поворот

Для sinusoidal PE можно получить `PE(pos + offset)` из `PE(pos)` через поворот каждой пары `(sin, cos)`. Для одной пары:

```text
new_even = cos(offset * omega) * even + sin(offset * omega) * odd
new_odd  = -sin(offset * omega) * even + cos(offset * omega) * odd
```

Это полезное свойство: преобразование зависит от относительного смещения `offset`, а не от абсолютной позиции `pos`.

> **Proof sketch: относительный сдвиг из формул сложения углов.**
>
> Пусть `a = pos * omega`, `b = offset * omega`.
>
> ```text
> sin(a + b) = sin(a)cos(b) + cos(a)sin(b)
> cos(a + b) = cos(a)cos(b) - sin(a)sin(b)
> ```
>
> Поэтому вектор `(sin(a), cos(a))` можно повернуть на угол `b` и получить `(sin(a+b), cos(a+b))`.
> Именно это проверяет mini-lab про `shift_sinusoidal_encoding`.

### Мини-лабораторная 6: реализуйте shift_sinusoidal_encoding

Напишите функцию `shift_sinusoidal_encoding`, которая получает PE-векторы и смещение `offset`, а возвращает векторы, соответствующие позициям `pos + offset`.

In [ ]:
# Мини-лабораторная 6
# Задача: реализовать относительный сдвиг через pairwise rotation.

def shift_sinusoidal_encoding(encoding: Tensor, offset: int | float, base: float = 10_000.0) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


<details>
<summary>Подсказка</summary>

Разделите `encoding` на `even = encoding[..., 0::2]` и `odd = encoding[..., 1::2]`. Для углов нужны частоты из `make_frequencies(d_model)` и `offset * frequencies`.
</details>

**После проверки:** объясните, почему ошибка не ровно нулевая, хотя формула математически точная.

In [ ]:
# Проверка мини-лабораторной 6
pe = make_sinusoidal_pe(max_len=80, d_model=16)
for offset in [1, 7, 32]:
    shifted = shift_sinusoidal_encoding(pe[:-offset], offset)
    target = pe[offset:]
    assert tuple(shifted.shape) == tuple(target.shape)
    max_error = (shifted - target).abs().max().item()
    print(f"offset={offset:>2}")
    report_check("shift max error", max_error, 2e-6)
    assert max_error < 2e-6


### Визуальный блок: поворот на циферблате

Возьмем одну пару измерений `(sin, cos)` и посмотрим координаты до и после сдвига. Это та самая 2D-интуиция из статьи: перейти от `pos` к `pos + k` значит повернуть точку на фиксированный угол.

In [ ]:
pair = 0
pos = 5
offset = 3
pe = make_sinusoidal_pe(max_len=16, d_model=8)
point = pe[pos, 2 * pair : 2 * pair + 2]
shifted_point = shift_sinusoidal_encoding(pe[pos : pos + 1], offset)[0, 2 * pair : 2 * pair + 2]
target_point = pe[pos + offset, 2 * pair : 2 * pair + 2]

print(f"pair={pair}, pos={pos}, offset={offset}")
print("point at pos:         ", point)
print("rotated point:        ", shifted_point)
print("target pos + offset:  ", target_point)
report_check("2D point rotation error", (shifted_point - target_point).abs().max().item(), 1e-6)


## 7. Toy RoPE: позиция как поворот query/key

Sinusoidal PE добавляется к token embeddings. RoPE использует похожую идею поворотов, но применяет позиционный поворот к векторам query/key внутри attention.

В этой лабораторной реализуем toy-версию RoPE для тензора формы `(batch, seq_len, d_model)`: для каждой позиции `pos` повернем каждую пару измерений на угол `pos * omega_i`.

> **Key distinction: additive PE vs RoPE.**
>
> Additive sinusoidal PE меняет входные token embeddings до attention.
> RoPE поворачивает query/key внутри attention, поэтому относительное расстояние напрямую влияет на dot product `q_i · k_j`.
>
> В хорошей реализации RoPE поворот сохраняет норму векторов. Поэтому norm preservation - обязательный sanity check.

### Мини-лабораторная 7: реализуйте apply_rope

Напишите `apply_rope(x)`. Проверки будут смотреть не на готовую функцию из библиотеки, а на свойства поворота:

- форма не меняется;
- норма каждого вектора сохраняется;
- позиция `0` не меняется, потому что угол равен нулю;
- ненулевые позиции действительно меняются.

In [ ]:
# Мини-лабораторная 7
# Задача: реализовать toy RoPE для x формы (batch, seq_len, d_model).

def apply_rope(x: Tensor, base: float = 10_000.0) -> Tensor:
    # здесь ваш код
    raise NotImplementedError


<details>
<summary>Подсказка</summary>

RoPE похож на shift, но `offset` заменяется на позицию каждого токена. Позиции имеют форму `(seq_len, 1)`, частоты `(d_model / 2,)`, углы `(seq_len, d_model / 2)`. Для batch можно сделать `cos.unsqueeze(0)` и `sin.unsqueeze(0)`.
</details>

**После проверки:** сравните additive PE и RoPE: где именно появляется позиционная информация?

In [ ]:
# Проверка мини-лабораторной 7
torch.manual_seed(11)
x_rope = torch.randn(2, 12, 16)
rotated = apply_rope(x_rope)

assert tuple(rotated.shape) == tuple(x_rope.shape)

norm_error = (rotated.norm(dim=-1) - x_rope.norm(dim=-1)).abs().max().item()
position0_error = (rotated[:, 0, :] - x_rope[:, 0, :]).abs().max().item()
changed_marker = (rotated[:, 1:, :] - x_rope[:, 1:, :]).abs().max().item()

print(f"norm preservation max error: {norm_error:.2e}")
print(f"position 0 identity max error: {position0_error:.2e}")
print(f"non-zero positions changed marker: {changed_marker:.2e}")

assert norm_error < 1e-6
assert position0_error < 1e-6
assert changed_marker > 1e-3

# Дополнительный marker: одинаковый content на разных позициях получает разные направления.
base = F.normalize(torch.randn(1, 1, 16), dim=-1)
repeated = base.repeat(1, 4, 1)
rope_repeated = apply_rope(repeated)
relative_dot_marker = (rope_repeated[:, 0, :] * rope_repeated[:, 3, :]).sum(dim=-1).item()
print(f"relative dot marker pos0-vs-pos3: {relative_dot_marker:.4f}")
assert relative_dot_marker < 1.0


### Визуальный блок: RoPE decay

Статья показывает long-range decay: если content одинаковый, то позиционный поворот сам по себе уменьшает dot product при росте расстояния. Ниже мы нормируем один и тот же вектор, применяем RoPE на разных позициях и считаем dot product с позицией 0.

In [ ]:
base_vector = torch.randn(1, 1, 32)
base_vector = F.normalize(base_vector, dim=-1)
repeated = base_vector.repeat(1, 64, 1)
rotated = apply_rope(repeated)
anchor = rotated[:, 0, :]

decay = {}
for distance in [0, 1, 2, 4, 8, 16, 32, 48]:
    decay[distance] = (anchor * rotated[:, distance, :]).sum(dim=-1).item()

ascii_curve(decay)
